In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import html
import re

# ---------- Setup ----------
chrome_options = Options()
chrome_options.add_argument("--headless")  # Uncomment to run headless
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 10)

# ---------- Input ----------
start_mc = 1679325
end_mc = 1690000
url = "https://safer.fmcsa.dot.gov/CompanySnapshot.aspx"

results = []

# ---------- Helper functions ----------
def get_field(label_text):
    """Extracts value from <li><label>Text</label><span class='dat'>Value</span></li>"""
    try:
        element = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, f"//li[label[contains(text(),'{label_text}')]]/span[@class='dat']")
            )
        )
        val = element.get_attribute("innerHTML")
        val = html.unescape(val.replace("<br>", " ").replace("\n", " ").replace("&nbsp;", " ")).strip()
        return val
    except:
        return "N/A"

def get_operating_authority_status():
    """Extract only 'AUTHORIZED FOR Property' from the field"""
    try:
        td_element = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, "//th[a[contains(text(),'Operating Authority Status:')]]/following-sibling::td")
            )
        )
        html_content = td_element.get_attribute("innerHTML")
        html_content = html.unescape(html_content)
        
        # Extract "AUTHORIZED FOR Property" using regex
        match = re.search(r'AUTHORIZED FOR Property', html_content)
        if match:
            return match.group(0)
        else:
            return "N/A"
    except:
        return "N/A"

def get_td_after_th(label_text):
    """Fallback for normal <td> fields"""
    try:
        td_element = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, f"//th[a[contains(text(),'{label_text}')]]/following-sibling::td")
            )
        )
        val = td_element.get_attribute("innerHTML")
        val = html.unescape(val.replace("<br>", " ").replace("\n", " ").replace("&nbsp;", " ")).strip()
        return val
    except:
        return "N/A"

# ---------- Scraping Loop ----------
for mc in range(start_mc, end_mc + 1):
    try:
        driver.get(url)

        # Select "MC/MX" radio button
        wait.until(EC.element_to_be_clickable((By.ID, "2"))).click()

        # Enter MC number
        search_box = wait.until(EC.presence_of_element_located((By.ID, "4")))
        search_box.clear()
        search_box.send_keys(str(mc))

        # Click Search
        driver.find_element(By.XPATH, "//input[@type='SUBMIT' and @value='Search']").click()

        # ---------- Check if MC exists ----------
        if driver.find_elements(By.XPATH, "//td[contains(text(),'No records matching')]"):
            print(f"⏭️ MC {mc} not found, skipping")
            continue

        # ---------- Get Operating Authority Status & Entity Type ----------
        status_text = get_operating_authority_status()
        entity_text = get_td_after_th("Entity Type:")

        if status_text != "AUTHORIZED FOR Property" or "CARRIER" not in entity_text:
            print(f"⏭️ MC {mc} skipped: Not authorized or not carrier")
            continue

        # ---------- Click "SMS Results" ----------
        sms_link = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'SMS Results')]")))
        sms_link.click()
        driver.switch_to.window(driver.window_handles[-1])

        # Click "Carrier Registration Details"
        reg_link = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'Carrier Registration Details')]")))
        reg_link.click()
        driver.switch_to.window(driver.window_handles[-1])

        # ---------- Scrape required fields ----------
        legal_name = get_field("Legal Name:")
        email = get_field("Email:")
        phone = get_field("Telephone:")
        address = get_field("Address:")

        # ---------- Save result ----------
        results.append({
            "MC": mc,
            "Legal Name": legal_name,
            "Email": email,
            "Phone": phone,
            "Address": address,
            "Operating Authority Status": status_text,
            "Entity Type": entity_text
        })

        print(f"✅ Scraped MC {mc}: {legal_name} | {status_text} | {entity_text}")

        # Close extra tabs
        while len(driver.window_handles) > 1:
            driver.close()
            driver.switch_to.window(driver.window_handles[0])

    except Exception as e:
        print(f"❌ MC {mc} failed: {e}")
        while len(driver.window_handles) > 1:
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
        continue

# ---------- Save to CSV ----------
df = pd.DataFrame(results)
df.to_csv("carrier_details.csv", index=False)

driver.quit()
print("Scraping complete. Data saved to carrier_details.csv")


There was an error managing chromedriver (error decoding response body); using driver found in the cache


KeyboardInterrupt: 